# Notebook 03 — Generate Answers with the Answer-Generator Models (Ollama)

## Purpose

This notebook asks each answer-generator model every question of the frozen German question set and stores the answers for the evaluation stage (Notebook 04). The generators are **inputs to the evaluation pipeline**, not the object of the thesis, so the notebook is built for a fair, reproducible and auditable comparison:

* **Same input for every model:** the same German question and the **complete Botschaft** (same text and `[PAGE n]` markers as in Notebook 02), the same instructions, the same decoding settings.
* **One answer per question per model**, in German. An English translation is stored in a separate column (`answer_en`) for reading only; it is never used for evaluation.
* **No evaluation criteria and no reference answer** are given to the generator.
* Everything needed to report the run (model digest, context size, seed, temperature, prompt version, token counts, durations) is saved with the answers.

## Models

| Key | Model | Ollama tag |
|---|---|---|
| `llama33_70b` | Llama 3.3 70B | `llama3.3:70b` |
| `apertus_70b` | Apertus 70B | `MichelRosselli/apertus:70b-instruct-2509-q4_k_m` |

`NUM_MODELS_TO_TEST` controls how many models of this list are run, in list order. It is set to **1** for now (Llama only). Set it to **2** after you have replaced `APERTUS_TAG_PLACEHOLDER` with the exact tag shown by `ollama list`.

Expected inputs and outputs:

```text
project/
├── data/
│   ├── extracted_text/                 # from Notebook 01
│   ├── questions/
│   │   └── question_set_frozen.csv     # frozen after the manual review of Notebook 02
│   └── answers/                        # written by this notebook
│       ├── answers_<model_key>.jsonl   # appended after every answer (resume-safe)
│       ├── translations_en_<model_key>.jsonl   # English translations (documentation only)
│       ├── answers_<model_key>.csv
│       ├── answers_all.csv
│       ├── answer_generation_run_config.json
│       └── answer_generation_call_log.csv
```

Run from the project root, or change `PROJECT_ROOT` below.

## 1. Configuration

All settings are in this cell and are saved to `answer_generation_run_config.json` at the end.

Notes on the choices:

* **Temperature 0 and a fixed seed** make the run as repeatable as Ollama allows. The same settings are used for every model.
* **`REQUEST_PAGE_REFERENCES = True`** asks the generator to name page numbers. In the framework these references are kept as a *diagnostic only*; groundedness is always checked by the evaluator against the Botschaft itself. Set it to `False` if you prefer answers without page references.
* **`PILOT_QUESTIONS_PER_DOCUMENT`**: set e.g. `2` to answer only the first two questions of every Botschaft, which is useful for a first test run and for the pilot of the evaluation framework. `None` runs everything.
* **`num_ctx` is computed per model** (Section 3) from the longest Botschaft and that model's own maximum context, because the models have different limits. The *content* of the prompt is identical for all models.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import re
import time
import unicodedata
import requests
import pandas as pd

# ------------------------------------------------------------
# Answer-generator models (run in list order)
# ------------------------------------------------------------
MODELS = [
    {"key": "llama33_70b", "label": "Llama 3.3 70B", "ollama_tag": "llama3.3:70b"},
    # Replace the placeholder with the exact tag from `ollama list` once the supervisor has loaded the model.
    {"key": "apertus_70b", "label": "Apertus 70B", "ollama_tag": "MichelRosselli/apertus:70b-instruct-2509-q4_k_m"},
]
NUM_MODELS_TO_TEST = 2               # 1 = Llama only; 2 = Llama and Apertus

# ------------------------------------------------------------
# Server and decoding settings (identical for every model)
# ------------------------------------------------------------
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_CHAT_URL = f"{OLLAMA_BASE_URL}/api/chat"
KEEP_ALIVE = "60m"
TEMPERATURE = 0.0
SEED = 42
THINK = None                           # None = do not send the flag (right for non-reasoning models)
MAX_OUTPUT_TOKENS = 2000
N_ANSWERS_PER_QUESTION = 1             # the thesis design uses one answer per question and model

# ------------------------------------------------------------
# Prompt and input
# ------------------------------------------------------------
REQUEST_PAGE_REFERENCES = True
PROMPT_VERSION = "answers-v1-fulldoc-2026-10"

# ------------------------------------------------------------
# English translation of the answers (documentation only, never used for evaluation)
# ------------------------------------------------------------
TRANSLATE_ANSWERS = True
TRANSLATION_MODEL_INDEX = 0            # index in ACTIVE_MODELS of the model that translates (0 = the first active model)
TRANSLATION_NUM_CTX = 8192
TRANSLATION_MAX_OUTPUT_TOKENS = 3000
TRANSLATION_PROMPT_VERSION = "translate-answers-v1-2026-10"

# ------------------------------------------------------------
# Context sizing
# ------------------------------------------------------------
TOKEN_CHARS_CONSERVATIVE = 3.0         # characters per token for sizing; German is usually 3.5-4
PROMPT_OVERHEAD_TOKENS = 2000          # instructions + question

# ------------------------------------------------------------
# Question file and pilot switch
# ------------------------------------------------------------
QUESTION_FILE_CANDIDATES = [           # first existing file is used
    "question_set_frozen.csv",
    "question_set_frozen.xlsx",
]
PILOT_QUESTIONS_PER_DOCUMENT = None    # e.g. 2 for a quick test run

# ------------------------------------------------------------
# Robustness and bookkeeping
# ------------------------------------------------------------
REQUEST_TIMEOUT_SECONDS = 3600
MAX_RETRIES = 3
BLIND_ID_SALT = "thesis-2026"          # used to create model-neutral answer IDs for the judge

PROJECT_ROOT = Path.cwd()
TEXT_DIR = PROJECT_ROOT / "../data" / "extracted_text"
QUESTION_DIR = PROJECT_ROOT / "../data" / "questions"
ANSWER_DIR = PROJECT_ROOT / "../data" / "answers"
ANSWER_DIR.mkdir(parents=True, exist_ok=True)

ACTIVE_MODELS = MODELS[:NUM_MODELS_TO_TEST]
assert 1 <= NUM_MODELS_TO_TEST <= len(MODELS), "NUM_MODELS_TO_TEST must be between 1 and the number of listed models"

print("Project root:", PROJECT_ROOT)
print("Models to run:", [m["label"] + " (" + m["ollama_tag"] + ")" for m in ACTIVE_MODELS])
print("Prompt version:", PROMPT_VERSION)

Project root: /home/bern/Documents/llm_trustworthiness/scripts
Models to run: ['Llama 3.3 70B (llama3.3:70b)', 'Apertus 70B (MichelRosselli/apertus:70b-instruct-2509-q4_k_m)']
Prompt version: answers-v1-fulldoc-2026-10


## 2. Load the frozen question set

The notebook uses the first existing file from `QUESTION_FILE_CANDIDATES`. If the file is the review workbook and it contains a filled `review_keep` column, only the rows marked as kept (`yes`, `y`, `ja`, `1`, `true`, `x`, `keep`) are used. The German question (`question_de`) is the only question text sent to the models.

In [2]:
def load_questions():
    for name in QUESTION_FILE_CANDIDATES:
        path = QUESTION_DIR / name
        if path.exists():
            if path.suffix.lower() == ".xlsx":
                df = pd.read_excel(path, sheet_name="Questions", dtype=str)
            else:
                df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
            return df.fillna(""), path
    raise FileNotFoundError(
        f"None of {QUESTION_FILE_CANDIDATES} found in {QUESTION_DIR}. "
        "Freeze the reviewed question set from Notebook 02 first."
    )

questions, QUESTION_PATH = load_questions()
required = {"question_id", "document_id", "question_de", "question_type"}
missing = required - set(questions.columns)
if missing:
    raise ValueError(f"The question file is missing required columns: {sorted(missing)}")

if "review_keep" in questions.columns and questions["review_keep"].str.strip().ne("").any():
    keep_values = {"y", "yes"}
    before = len(questions)
    questions = questions[questions["review_keep"].str.strip().str.lower().isin(keep_values)].copy()
    print(f"Applied review_keep filter: {before} -> {len(questions)} questions")

questions = questions[questions["question_de"].str.strip() != ""].reset_index(drop=True)
if questions["question_id"].duplicated().any():
    raise ValueError("question_id values must be unique.")

if PILOT_QUESTIONS_PER_DOCUMENT:
    questions = questions.groupby("document_id", sort=False).head(PILOT_QUESTIONS_PER_DOCUMENT).reset_index(drop=True)
    print(f"PILOT MODE: {PILOT_QUESTIONS_PER_DOCUMENT} question(s) per document")

print("Question file:", QUESTION_PATH)
print("Questions to answer:", len(questions))
display(questions.groupby(["document_id", "question_type"]).size().rename("n").reset_index())

Applied review_keep filter: 83 -> 75 questions
Question file: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_frozen.xlsx
Questions to answer: 75


,document_id,question_type,n
0,13_ahv_rente,challenging,4
1,13_ahv_rente,comparative,1
2,13_ahv_rente,explanatory,3
3,13_ahv_rente,factual,4
4,nachhaltigkeitsinitiative,challenging,3
5,nachhaltigkeitsinitiative,comparative,3
6,nachhaltigkeitsinitiative,explanatory,3
7,nachhaltigkeitsinitiative,factual,2
8,neutralitaetsinitiative,challenging,4
9,neutralitaetsinitiative,comparative,3


## 3. Load the Botschaften and check that every model can take them

The documents are assembled **exactly as in Notebook 02** (page text with `[PAGE n]` markers), so generators and judge see the same source text.

For every active model, the notebook reads the installed tag, digest, quantisation and maximum context length from Ollama, then checks that the longest Botschaft plus instructions and the output budget fits. If it does not fit, the notebook **stops** instead of letting Ollama silently truncate the document, because a truncated input would make the answers incomparable. In that case the options are a model with a longer context or a different input mode (for example retrieved passages); that would then have to be applied identically to all generators.

Memory note: for a 70B Llama model the key-value cache grows by roughly 0.3 MB per token, so a 100,000-token prompt needs about 30 GB on top of the 42 GB of weights. Watch `nvidia-smi` during the first call.

In [3]:
def load_json_document(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    pages = []
    for page in payload.get("pages", []):
        text = (page.get("text") or "").strip()
        if text:
            pages.append({"page": int(page["page"]) if page.get("page") is not None else None, "text": text})
    return {
        "document_id": payload.get("selection_key") or path.stem,
        "title": payload.get("title") or path.stem,
        "date": payload.get("date"),
        "pages": pages,
    }

def parse_txt_pages(path: Path) -> dict:
    text = path.read_text(encoding="utf-8", errors="replace")
    title_match = re.search(r"^TITLE:\s*(.+)$", text, flags=re.MULTILINE)
    matches = list(re.finditer(r"===== PAGE (\d+) =====", text))
    pages = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        page_text = text[m.end():end].strip()
        if page_text:
            pages.append({"page": int(m.group(1)), "text": page_text})
    if not pages:
        pages = [{"page": None, "text": text}]
    return {"document_id": path.stem, "title": title_match.group(1).strip() if title_match else path.stem,
            "date": None, "pages": pages}

def assemble_text(pages):
    return "\n\n".join((f"[PAGE {p['page']}]\n" if p["page"] is not None else "") + p["text"] for p in pages)

if not TEXT_DIR.exists():
    raise FileNotFoundError(f"Could not find {TEXT_DIR}. Run Notebook 01 first.")

all_docs, used = {}, set()
for jp in sorted(TEXT_DIR.glob("*.json")):
    try:
        d = load_json_document(jp)
        if d["pages"]:
            all_docs[d["document_id"]] = d
            used.add(jp.stem)
    except (json.JSONDecodeError, OSError) as exc:
        print(f"WARNING: could not read {jp.name}: {exc}")
for tp in sorted(TEXT_DIR.glob("*.txt")):
    if tp.stem not in used:
        d = parse_txt_pages(tp)
        if d["pages"]:
            all_docs[d["document_id"]] = d

needed_ids = set(questions["document_id"])
unknown = needed_ids - set(all_docs)
if unknown:
    raise RuntimeError(f"Questions refer to documents that were not found in {TEXT_DIR}: {sorted(unknown)}")

documents = {k: v for k, v in all_docs.items() if k in needed_ids}
for d in documents.values():
    d["text"] = assemble_text(d["pages"])
    d["est_tokens"] = int(len(d["text"]) / TOKEN_CHARS_CONSERVATIVE)

longest = max(d["est_tokens"] for d in documents.values())
display(pd.DataFrame([{"document_id": k, "title": v["title"], "pages": len(v["pages"]),
                       "characters": len(v["text"]), "est_tokens_conservative": v["est_tokens"]}
                      for k, v in documents.items()]))

# ---------- per-model checks ----------
def ollama_get(path):
    r = requests.get(f"{OLLAMA_BASE_URL}{path}", timeout=60)
    r.raise_for_status()
    return r.json()

def ollama_post(path, payload, timeout=120):
    r = requests.post(f"{OLLAMA_BASE_URL}{path}", json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()

installed = {m["name"]: m for m in ollama_get("/api/tags").get("models", [])}
try:
    OLLAMA_VERSION = ollama_get("/api/version").get("version", "unknown")
except Exception:
    OLLAMA_VERSION = "unknown"

for m in ACTIVE_MODELS:
    tag = m["ollama_tag"]
    if "PLACEHOLDER" in tag.upper():
        raise RuntimeError(
            f"The tag for {m['label']} is still a placeholder ({tag!r}). Replace it with the exact name from "
            "`ollama list` in the MODELS cell, or set NUM_MODELS_TO_TEST to exclude this model."
        )
    if tag not in installed:
        raise RuntimeError(f"Model {tag!r} is not installed. Installed: {sorted(installed)}")
    show = ollama_post("/api/show", {"model": tag})
    info = show.get("model_info", {})
    ctx_keys = [k for k in info if k.endswith(".context_length")]
    m["digest"] = installed[tag].get("digest", "")
    m["max_ctx"] = int(info[ctx_keys[0]]) if ctx_keys else None
    m["parameter_size"] = show.get("details", {}).get("parameter_size")
    m["quantization"] = show.get("details", {}).get("quantization_level")
    m["capabilities"] = show.get("capabilities")
    m["model_defaults"] = show.get("parameters")
    if m["max_ctx"] is None:
        raise RuntimeError(f"Could not read the maximum context length of {tag} from `ollama show`.")

    needed_tokens = longest + PROMPT_OVERHEAD_TOKENS + MAX_OUTPUT_TOKENS
    if needed_tokens > m["max_ctx"]:
        raise RuntimeError(
            f"{m['label']}: the longest Botschaft needs about {needed_tokens:,} tokens (document {longest:,} + "
            f"overhead + output) but the model's maximum context is {m['max_ctx']:,}. "
            "Do not let Ollama truncate: use a longer-context model or change the input mode for ALL generators."
        )
    m["num_ctx"] = min(m["max_ctx"], int(math.ceil(needed_tokens / 4096) * 4096))
    print(f"{m['label']:16s} tag={tag}  digest={m['digest'][:12]}  {m['parameter_size']}  {m['quantization']}  "
          f"max_ctx={m['max_ctx']:,}  num_ctx used={m['num_ctx']:,}")
print("Ollama version:", OLLAMA_VERSION)

,document_id,title,pages,characters,est_tokens_conservative
0,13_ahv_rente,Botschaft zur Volksinitiative «Für ein bessere...,28,74703,24901
1,nachhaltigkeitsinitiative,Botschaft zur Volksinitiative «Keine 10-Millio...,110,299015,99671
2,neutralitaetsinitiative,Botschaft zur Volksinitiative «Wahrung der sch...,30,80586,26862
3,steuergerechtigkeits_initiative,Botschaft zur Volksinitiative «Für eine zivils...,108,274951,91650
4,stopfleber_initiative,Botschaft zur Volksinitiative «Ja zum Importve...,16,44034,14678
5,umweltverantwortungsinitiative,Botschaft zur Volksinitiative «Für eine verant...,20,50122,16707


Llama 3.3 70B    tag=llama3.3:70b  digest=a6eb4748fd29  70.6B  Q4_K_M  max_ctx=131,072  num_ctx used=106,496


RuntimeError: Apertus 70B: the longest Botschaft needs about 103,671 tokens (document 99,671 + overhead + output) but the model's maximum context is 65,536. Do not let Ollama truncate: use a longer-context model or change the input mode for ALL generators.

## 4. Prompt

The instructions are in German, the document is placed in the **system message** and the question alone in the **user message**. Because the system message is identical for all questions of a Botschaft, Ollama can reuse the already-processed document and only has to read the new question. Questions are therefore asked document by document.

The instructions tell the model to answer only from the Botschaft, to say so when the information is missing, to attribute positions to the actors who hold them, and not to give its own political opinion or a voting recommendation. They deliberately do **not** mention the evaluation criteria.

In [5]:
def build_system_prompt(doc):
    rules = [
        "Beantworte die Frage ausschliesslich auf Grundlage der Botschaft. Verwende kein Wissen von ausserhalb der Botschaft.",
        "Antworte auf Deutsch, präzise und vollständig, aber ohne unnötige Ausführlichkeit.",
        "Wenn die Botschaft die gefragte Information nicht enthält, sage das ausdrücklich.",
        "Schreibe Positionen und Bewertungen den Akteuren zu, die sie vertreten (zum Beispiel Bundesrat, Initiativkomitee, Parlament). "
        "Gib keine eigene politische Meinung und keine Abstimmungsempfehlung ab.",
    ]
    if REQUEST_PAGE_REFERENCES:
        rules.append("Nenne die Seitenzahlen der verwendeten Stellen in der Form (S. 12).")
    numbered = "\n".join(f"{i}. {r}" for i, r in enumerate(rules, start=1))
    return (
        "Du bist ein sachlicher Assistent, der Fragen von Stimmberechtigten zu einer offiziellen Botschaft "
        "des Schweizer Bundesrates beantwortet. Dir liegt die vollständige Botschaft vor; die Seiten sind mit "
        "[PAGE n] markiert.\n\n"
        f"Regeln:\n{numbered}\n\n"
        f"DOKUMENTTITEL: {doc['title']}\n"
        f"DATUM: {doc.get('date') or 'unbekannt'}\n\n"
        f"<BOTSCHAFT>\n{doc['text']}\n</BOTSCHAFT>"
    )

def build_user_prompt(question_de):
    return f"Frage: {question_de.strip()}"

# show the prompt skeleton (without the long document) once
example_doc = next(iter(documents.values()))
skeleton = build_system_prompt({**example_doc, "text": "[… vollständiger Text der Botschaft …]"})
print(skeleton)
print("\n--- user message ---\n" + build_user_prompt("<Frage aus dem Fragenset>"))

Du bist ein sachlicher Assistent, der Fragen von Stimmberechtigten zu einer offiziellen Botschaft des Schweizer Bundesrates beantwortet. Dir liegt die vollständige Botschaft vor; die Seiten sind mit [PAGE n] markiert.

Regeln:
1. Beantworte die Frage ausschliesslich auf Grundlage der Botschaft. Verwende kein Wissen von ausserhalb der Botschaft.
2. Antworte auf Deutsch, präzise und vollständig, aber ohne unnötige Ausführlichkeit.
3. Wenn die Botschaft die gefragte Information nicht enthält, sage das ausdrücklich.
4. Schreibe Positionen und Bewertungen den Akteuren zu, die sie vertreten (zum Beispiel Bundesrat, Initiativkomitee, Parlament). Gib keine eigene politische Meinung und keine Abstimmungsempfehlung ab.
5. Nenne die Seitenzahlen der verwendeten Stellen in der Form (S. 12).

DOKUMENTTITEL: Botschaft zur Volksinitiative «Für ein besseres Leben im Alter (Initiative für eine 13. AHV-Rente)»
DATUM: 2022-06-21

<BOTSCHAFT>
[… vollständiger Text der Botschaft …]
</BOTSCHAFT>

--- user m

## 5. Helper functions

`chat_ollama` sends one request with the model-specific `num_ctx`, the shared decoding settings and the seed, retries on errors, strips any `<think>` block, and records token counts and durations. An answer that stops because it hit the output limit is **kept but flagged** (`truncated = True`) and is not retried, so you can decide how to treat it.

In [6]:
CALL_LOG = []

def strip_think(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def chat_ollama(model, system_prompt, user_prompt, seed, label):
    payload = {
        "model": model["ollama_tag"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": model["num_ctx"],
            "num_predict": MAX_OUTPUT_TOKENS,
            "seed": seed,
        },
    }
    if THINK is not None:
        payload["think"] = THINK

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            started = time.time()
            r = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=(30, REQUEST_TIMEOUT_SECONDS))
            r.raise_for_status()
            result = r.json()
            text = strip_think((result.get("message") or {}).get("content", ""))
            stats = {
                "label": label,
                "model_key": model["key"],
                "prompt_eval_count": result.get("prompt_eval_count"),
                "eval_count": result.get("eval_count"),
                "done_reason": result.get("done_reason"),
                "total_seconds": round(result.get("total_duration", 0) / 1e9, 1),
                "wall_seconds": round(time.time() - started, 1),
            }
            CALL_LOG.append(stats)
            if not text:
                raise RuntimeError("empty answer")
            return text, stats
        except Exception as exc:
            last_error = exc
            print(f"[{label}] request failed (attempt {attempt}/{MAX_RETRIES}): {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)
    raise RuntimeError(f"[{label}] failed after {MAX_RETRIES} attempts: {last_error}")

def blind_id(answer_id: str) -> str:
    return hashlib.sha1(f"{BLIND_ID_SALT}|{answer_id}".encode("utf-8")).hexdigest()[:10]

def load_done_ids(path: Path):
    done = set()
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    done.add(json.loads(line)["answer_id"])
                except (json.JSONDecodeError, KeyError):
                    pass
    return done

def unload_model(model):
    # free GPU memory for the next stage (keep_alive = 0 unloads the model)
    try:
        requests.post(f"{OLLAMA_BASE_URL}/api/generate",
                      json={"model": model["ollama_tag"], "keep_alive": 0}, timeout=120)
    except Exception as exc:
        print("Could not unload", model["ollama_tag"], exc)

## 6. Generate the answers

Models run one after the other. Answers are appended to `answers_<model_key>.jsonl` **after every single answer**, so an interrupted run can simply be started again: answers that already exist are skipped. To redo answers on purpose, delete the corresponding JSONL file (or change `PROMPT_VERSION`).

Each record keeps the model tag and digest, the settings, the token counts, a `truncated` flag and a model-neutral `blind_id` that Notebook 04 can show to the judge instead of the model name.

In [7]:
questions_by_doc = {doc_id: grp for doc_id, grp in questions.groupby("document_id", sort=False)}
run_started = pd.Timestamp.now(tz="UTC")

for model in ACTIVE_MODELS:
    out_path = ANSWER_DIR / f"answers_{model['key']}.jsonl"
    done = load_done_ids(out_path)
    total = len(questions) * N_ANSWERS_PER_QUESTION
    print(f"\n=== {model['label']} ({model['ollama_tag']}) — {len(done)} of {total} answers already on disk ===")
    t0, n_new = time.time(), 0

    for doc_id, grp in questions_by_doc.items():
        doc = documents[doc_id]
        system_prompt = build_system_prompt(doc)          # identical prefix for all questions of this document
        for _, q in grp.iterrows():
            for run in range(1, N_ANSWERS_PER_QUESTION + 1):
                answer_id = f"{model['key']}__{q['question_id']}__r{run}"
                if answer_id in done:
                    continue
                answer, stats = chat_ollama(
                    model, system_prompt, build_user_prompt(q["question_de"]),
                    seed=SEED + run - 1, label=answer_id,
                )
                record = {
                    "answer_id": answer_id,
                    "blind_id": blind_id(answer_id),
                    "question_id": q["question_id"],
                    "document_id": doc_id,
                    "question_type": q["question_type"],
                    "question_de": q["question_de"],
                    "model_key": model["key"],
                    "model_label": model["label"],
                    "ollama_tag": model["ollama_tag"],
                    "model_digest": model["digest"][:12],
                    "run": run,
                    "answer_de": answer,
                    "truncated": stats["done_reason"] == "length",
                    "prompt_eval_count": stats["prompt_eval_count"],
                    "eval_count": stats["eval_count"],
                    "wall_seconds": stats["wall_seconds"],
                    "num_ctx": model["num_ctx"],
                    "temperature": TEMPERATURE,
                    "seed": SEED + run - 1,
                    "prompt_version": PROMPT_VERSION,
                    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                }
                with out_path.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                done.add(answer_id)
                n_new += 1
                if n_new % 5 == 0 or n_new == 1:
                    rate = (time.time() - t0) / n_new
                    remaining = max(total - len(done), 0)
                    print(f"  {len(done)}/{total} answers  (last {stats['wall_seconds']}s, "
                          f"≈{rate/60:.1f} min/answer avg, ≈{remaining*rate/60:.0f} min left)")

    unload_model(model)
    print(f"Finished {model['label']}: {n_new} new answers in {(time.time()-t0)/60:.1f} min")


=== Llama 3.3 70B (llama3.3:70b) — 0 of 75 answers already on disk ===
  1/75 answers  (last 48.9s, ≈0.8 min/answer avg, ≈60 min left)
  5/75 answers  (last 3.2s, ≈0.3 min/answer avg, ≈18 min left)
  10/75 answers  (last 9.5s, ≈0.2 min/answer avg, ≈11 min left)
  15/75 answers  (last 18.9s, ≈0.4 min/answer avg, ≈24 min left)
  20/75 answers  (last 8.9s, ≈0.4 min/answer avg, ≈21 min left)
  30/75 answers  (last 2.8s, ≈0.3 min/answer avg, ≈14 min left)
  35/75 answers  (last 4.7s, ≈0.3 min/answer avg, ≈12 min left)
  40/75 answers  (last 17.2s, ≈0.4 min/answer avg, ≈12 min left)
  45/75 answers  (last 18.3s, ≈0.3 min/answer avg, ≈10 min left)
  55/75 answers  (last 2.4s, ≈0.3 min/answer avg, ≈6 min left)
  70/75 answers  (last 3.5s, ≈0.3 min/answer avg, ≈1 min left)
  75/75 answers  (last 8.0s, ≈0.3 min/answer avg, ≈0 min left)
Finished Llama 3.3 70B: 75 new answers in 20.1 min


## 6b. Translate the answers into English (documentation only)

The German answer (`answer_de`) is the answer that is evaluated. The English translation is stored in its **own column** (`answer_en`) and in its own file (`translations_en_<model_key>.jsonl`), so that the German master record is never changed. It is meant for the author's own reading, the human validation and the thesis examples. **Notebook 04 does not read it.**

The translation is literal and adds nothing. A small glossary makes sure that the official English terms of the Swiss government are used, in particular **Botschaft = "dispatch"** (not "message"). The same translation settings apply to the answers of every model. Translations that already exist are skipped when the cell is run again. Set `TRANSLATE_ANSWERS = False` to skip the step.

In [8]:
TRANSLATION_GLOSSARY = [
    ("Botschaft", "dispatch (the official English term for a Botschaft of the Federal Council; never 'message')"),
    ("Bundesrat", "Federal Council"),
    ("Bundesversammlung", "Federal Assembly"),
    ("Volksinitiative", "popular initiative"),
    ("Initiativkomitee", "initiative committee"),
    ("Gegenentwurf", "counter-proposal"),
    ("Stimmberechtigte", "eligible voters"),
]

def translation_system_prompt():
    glossary = "\n".join(f"- {de} -> {en}" for de, en in TRANSLATION_GLOSSARY)
    return (
        "You are a precise translator from German into English. Translate the text faithfully and literally. "
        "Do not add, remove, correct, summarise or comment on anything, even if the text seems wrong or incomplete. "
        "Keep numbers, names, dates, page references such as (S. 12) -> (p. 12), and paragraph structure. "
        "Use these official terms:\n" + glossary + "\n"
        "Return only the English translation."
    )

def translations_path(model):
    return ANSWER_DIR / f"translations_en_{model['key']}.jsonl"

def load_translations(model):
    out = {}
    path = translations_path(model)
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    rec = json.loads(line)
                    out[rec["answer_id"]] = rec["answer_en"]
                except (json.JSONDecodeError, KeyError):
                    pass
    return out

def translate_text(model, text, label):
    payload = {
        "model": model["ollama_tag"],
        "messages": [{"role": "system", "content": translation_system_prompt()},
                     {"role": "user", "content": text}],
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "options": {"temperature": 0.0, "num_ctx": TRANSLATION_NUM_CTX,
                    "num_predict": TRANSLATION_MAX_OUTPUT_TOKENS, "seed": SEED},
    }
    if THINK is not None:
        payload["think"] = THINK
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=(30, REQUEST_TIMEOUT_SECONDS))
            r.raise_for_status()
            result = r.json()
            out = strip_think((result.get("message") or {}).get("content", ""))
            if not out:
                raise RuntimeError("empty translation")
            if result.get("done_reason") == "length":
                print(f"[{label}] WARNING: translation hit the output limit")
            return out
        except Exception as exc:
            last_error = exc
            print(f"[{label}] translation failed (attempt {attempt}/{MAX_RETRIES}): {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)
    raise RuntimeError(f"[{label}] translation failed: {last_error}")

if TRANSLATE_ANSWERS:
    translator = ACTIVE_MODELS[TRANSLATION_MODEL_INDEX]
    print(f"Translating with {translator['label']} ({translator['ollama_tag']})")
    t0 = time.time()
    for model in ACTIVE_MODELS:
        gen_path = ANSWER_DIR / f"answers_{model['key']}.jsonl"
        if not gen_path.exists():
            continue
        existing = load_translations(model)
        n_new = 0
        for line in gen_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            rec = json.loads(line)
            if rec["answer_id"] in existing:
                continue
            english = translate_text(translator, rec["answer_de"], rec["answer_id"])
            with translations_path(model).open("a", encoding="utf-8") as f:
                f.write(json.dumps({"answer_id": rec["answer_id"], "answer_en": english,
                                    "translator_tag": translator["ollama_tag"],
                                    "translator_digest": translator["digest"][:12],
                                    "translation_prompt_version": TRANSLATION_PROMPT_VERSION}, ensure_ascii=False) + "\n")
            existing[rec["answer_id"]] = english
            n_new += 1
        print(f"{model['label']}: {n_new} new translations, {len(existing)} in total ({(time.time()-t0)/60:.1f} min)")
    unload_model(translator)
else:
    print("TRANSLATE_ANSWERS is False: no English translations created.")

Translating with Llama 3.3 70B (llama3.3:70b)
Llama 3.3 70B: 75 new translations, 75 in total (6.2 min)


## 7. Save the outputs

The JSONL files are the master record. This cell exports one CSV per model and a combined `answers_all.csv`, plus the run configuration and the call log.

In [9]:
frames = []
for model in ACTIVE_MODELS:
    path = ANSWER_DIR / f"answers_{model['key']}.jsonl"
    if not path.exists():
        continue
    df = pd.read_json(path, lines=True, dtype=False)
    df["answer_en"] = df["answer_id"].map(load_translations(model)).fillna("")
    cols = list(df.columns); cols.remove("answer_en"); cols.insert(cols.index("answer_de") + 1, "answer_en")
    df = df[cols]
    df.to_csv(ANSWER_DIR / f"answers_{model['key']}.csv", index=False, encoding="utf-8-sig")
    frames.append(df)

if not frames:
    raise RuntimeError("No answers found on disk.")
answers_all = pd.concat(frames, ignore_index=True)
answers_all.to_csv(ANSWER_DIR / "answers_all.csv", index=False, encoding="utf-8-sig")
if CALL_LOG:
    pd.DataFrame(CALL_LOG).to_csv(ANSWER_DIR / "answer_generation_call_log.csv", index=False)

run_config = {
    "question_file": str(QUESTION_PATH.relative_to(PROJECT_ROOT)),
    "n_questions": int(len(questions)),
    "pilot_questions_per_document": PILOT_QUESTIONS_PER_DOCUMENT,
    "n_answers_per_question": N_ANSWERS_PER_QUESTION,
    "models": [{k: m.get(k) for k in ["key", "label", "ollama_tag", "digest", "parameter_size", "quantization",
                                      "capabilities", "max_ctx", "num_ctx", "model_defaults"]} for m in ACTIVE_MODELS],
    "temperature": TEMPERATURE, "seed": SEED, "think": THINK, "max_output_tokens": MAX_OUTPUT_TOKENS,
    "request_page_references": REQUEST_PAGE_REFERENCES, "prompt_version": PROMPT_VERSION,
    "input_mode": "full document in system message, [PAGE n] markers",
    "ollama_version": OLLAMA_VERSION,
    "answer_translation": ({"model": ACTIVE_MODELS[TRANSLATION_MODEL_INDEX]["ollama_tag"], "prompt_version": TRANSLATION_PROMPT_VERSION,
                            "glossary": TRANSLATION_GLOSSARY, "used_for_evaluation": False} if TRANSLATE_ANSWERS else None),
    "documents": [{"document_id": k, "pages": len(v["pages"]), "characters": len(v["text"])} for k, v in documents.items()],
    "run_started_utc": run_started.isoformat(),
    "run_finished_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
(ANSWER_DIR / "answer_generation_run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", ANSWER_DIR / "answers_all.csv", f"({len(answers_all)} answers)")
display(answers_all[["answer_id", "question_type", "question_de", "answer_de"]].head(5))

Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/answers/answers_all.csv (75 answers)


,answer_id,question_type,question_de,answer_de
0,llama33_70b__13_ahv_rente_Q02__r1,comparative,Welche unterschiedlichen Auswirkungen hätte di...,Antwort: Die Volksinitiative «Für ein besseres...
1,llama33_70b__13_ahv_rente_Q04__r1,explanatory,Welche Auswirkungen hätte die Annahme der Volk...,Antwort: Laut Botschaft würde sich das Umlaged...
2,llama33_70b__13_ahv_rente_Q05__r1,explanatory,Wie begründet das Initiativkomitee die Notwend...,Das Initiativkomitee begründet die Notwendigke...
3,llama33_70b__13_ahv_rente_Q06__r1,explanatory,Wie sieht die Botschaft die Auswirkungen der I...,Die Botschaft betrachtet die Auswirkungen der ...
4,llama33_70b__13_ahv_rente_Q07__r1,factual,Welchen Betrag an zusätzlichen Ausgaben für di...,Laut Botschaft prognostiziert der Bundesrat zu...


## 8. Diagnostics

Check these before moving on to the evaluation notebook. In particular, read a handful of answers yourself: the evaluation pipeline can only be calibrated if you know what typical generator output looks like (wrong, missing or invented content, refusals, language mix-ups).

In [10]:
d = answers_all.copy()
d["answer_chars"] = d["answer_de"].str.len()
d["has_page_ref"] = d["answer_de"].str.contains(r"\bS\.\s*\d+", regex=True)
d["says_not_in_document"] = d["answer_de"].str.contains(
    r"nicht (?:enthalten|erwähnt|angegeben|zu finden)|keine (?:Angaben|Informationen)", case=False, regex=True)

summary = d.groupby("model_label").agg(
    answers=("answer_id", "count"),
    truncated=("truncated", "sum"),
    median_chars=("answer_chars", "median"),
    share_with_page_ref=("has_page_ref", "mean"),
    share_not_in_document=("says_not_in_document", "mean"),
    median_seconds=("wall_seconds", "median"),
).round(2)
display(summary)

print("Answers per document and model:")
display(d.groupby(["document_id", "model_label"]).size().unstack(fill_value=0))

if d["truncated"].any():
    print("\nWARNING: some answers hit the output limit. Raise MAX_OUTPUT_TOKENS and regenerate those answers "
          "(delete them from the JSONL file) if completeness matters for them.")
    display(d.loc[d["truncated"], ["answer_id", "answer_chars"]])

if CALL_LOG:
    log = pd.DataFrame(CALL_LOG)
    print("\nLargest prompt_eval_count this run:", int(log["prompt_eval_count"].max()),
          "| num_ctx:", {m["key"]: m["num_ctx"] for m in ACTIVE_MODELS})

,answers,truncated,median_chars,share_with_page_ref,share_not_in_document,median_seconds
model_label,,,,,,
Llama 3.3 70B,75,0,629.0,0.83,0.01,9.0


Answers per document and model:


model_label,Llama 3.3 70B
document_id,
13_ahv_rente,12
nachhaltigkeitsinitiative,11
neutralitaetsinitiative,13
steuergerechtigkeits_initiative,12
stopfleber_initiative,13
umweltverantwortungsinitiative,14



Largest prompt_eval_count this run: 94580 | num_ctx: {'llama33_70b': 106496}


## 9. Notes for the next stage and for the thesis

* **Fair comparison:** when Apertus is added, keep `PROMPT_VERSION`, `TEMPERATURE`, `SEED`, `MAX_OUTPUT_TOKENS` and the question file unchanged and only set `NUM_MODELS_TO_TEST = 2` (after replacing the placeholder tag). The notebook skips the Llama answers that already exist and generates only the Apertus ones.
* **Blinding:** Notebook 04 should show the judge only `blind_id`, `question_de`, `answer_de` and the Botschaft, never the model name (source framing can change LLM judgments).
* **Reporting:** `answer_generation_run_config.json` contains the model digests, quantisation, context sizes, decoding settings and the Ollama version for the computing-environment section of the thesis.
* If the prompt text, the question file or a model changes, change `PROMPT_VERSION` and regenerate.
* The English translation (`answer_en`) is documentation only. Notebook 04 evaluates `answer_de` and never sees `answer_en`.
* This notebook does not evaluate answers; that is Notebook 04.